# SymFormer / TBX11K — Colab Runbook (Core-Method PoC)

> **This is the Colab PoC (legacy).** It is the record of how the results in
> [`report.md`](../report.md) were produced: the TB subset only, torchvision only, and an
> architecture shaped entirely by Colab's constraints (ephemeral `/content`, a 15 GB Drive
> quota, weights kept off Drive because Colab's mount turns deletes into quota-consuming Trash
> moves).
>
> **For a local machine use [`local_runbook.ipynb`](local_runbook.ipynb)** - it runs the *full*
> 11,200-image dataset, adds the stage-2 classifier and the all-images evaluation mode, offers
> mmdetection alongside torchvision, and has a proper trainer (progress bar, metrics, plots,
> resume). Drop it on a PC and Run All.

---


Executes the phases in `plan.md` on a free Colab **T4**. Run cells top-to-bottom.

**Stack: torch + torchvision only.** No mmcv/mmdetection — OpenMMLab ships wheels only up to
~torch 2.1 / Python 3.11 and Colab is on Python 3.12, so `mim install mmcv` falls into a failing
source build. torchvision gives the same ResNet-50 + FPN + RetinaNet architecture with **zero
installs**. Our SAS block is pure torch and is unchanged.

**Storage split** (free Drive is 15GB; the Colab VM has ~100GB of ephemeral disk):

| What | Where | Size |
|---|---|---|
| code / docs | **GitHub** → `/content/FOP` | ~200 KB |
| raw TBX11K | **`/content`** (ephemeral) | ~tens of GB |
| **checkpoints** | **`/content/work`** (ephemeral — never Drive) | ~300 MB transient |
| compact TB-only 512² dataset | **Drive** | ~250–400 MB |
| **logs** (`train_log.jsonl` / `eval_log.jsonl`) — *the results* | **Drive** | a few KB |

**Weights never go on Drive.** Colab's Drive mount turns every delete into a move to Drive's
**Trash**, which keeps counting against your 15GB quota for 30 days — so writing ~300 MB per epoch
and pruning the previous one silently trashes **~7 GB per 24-epoch run**. A full run is only
~15–20 min, and every AP/AP50 lands in the logs, so the **logs are the results** and the weights are
disposable. Total Drive usage: **~350 MB**.

Training auto-resumes **within** a session. If the session dies, the ~20 min retrain is far cheaper
than the quota — just re-run the cell.
See `paper.md` (method + target numbers), `CLAUDE.md` (scope/recipe), `limitations.md` (caveats).

## Phase 1 — environment

In [ ]:
!nvidia-smi   # confirm a GPU (ideally a T4) is attached; if none, change runtime or retry later

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
# ---- paths ----
# Free Drive is only 15GB, but the Colab VM has ~100GB of EPHEMERAL local disk, so:
#   raw TBX11K (tens of GB) -> /content       (download, prep, then let it die with the session)
#   checkpoints (~300MB/ep) -> /content/work  (NEVER Drive — see below)
#   compact 512 dataset + LOGS -> Drive       (the only things worth persisting)
#
# Why weights never go on Drive: Colab's Drive mount turns every delete into a move to Drive's
# TRASH, and trashed files keep counting against your quota for 30 days. We write ~300MB per epoch
# and prune the previous one, so a 24-epoch run silently trashes ~7GB. Meanwhile a full run is only
# ~15-20min, and every AP/AP50 already lands in the logs (a few KB) — so the LOGS are the results
# and the weights are disposable. Expected Drive usage: ~350MB total.
REPO_URL = 'https://github.com/Manas-Maahir/FOP.git'
PROJECT  = '/content/FOP'
DRIVE    = '/content/drive/MyDrive'
DATA_RAW = '/content/TBX11K'              # raw TBX11K — EPHEMERAL (fixed up after extraction)
DATA_512 = f'{DRIVE}/tbx11k_tb512'        # compact TB-only 512 dataset (built in Phase 3)
WORK     = '/content/work'                # checkpoints — EPHEMERAL, dies with the session
RUNS     = f'{DRIVE}/tb_runs'             # logs only (KB) — these are the results
os.makedirs(WORK, exist_ok=True)
os.makedirs(RUNS, exist_ok=True)
print('project        :', PROJECT)
print('raw (temp)     :', DATA_RAW)
print('compact (Drive):', DATA_512)
print('weights (temp) :', WORK)
print('logs (Drive)   :', RUNS)

In [ ]:
# Pull the code from GitHub. Re-running this in a later session picks up anything pushed since.
if os.path.isdir(PROJECT):
    !cd {PROJECT} && git pull --ff-only
else:
    !git clone {REPO_URL} {PROJECT}
%cd {PROJECT}
!ls

In [ ]:
# No mmcv/mmdet. torch + torchvision ship with Colab; pycocotools usually does too.
# DO NOT `pip install -U openmim` — it downgrades setuptools and breaks pkg_resources on Py 3.12.
import torch, torchvision, sys
print('python     ', sys.version.split()[0])
print('torch      ', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')
print('torchvision', torchvision.__version__)
try:
    import pycocotools; print('pycocotools OK')
except ImportError:
    !pip install -q pycocotools
    import pycocotools; print('pycocotools installed')
open('requirements.lock.txt','w').write(
    f"python=={sys.version.split()[0]}\ntorch=={torch.__version__}\n"
    f"torchvision=={torchvision.__version__}\n")
print('wrote requirements.lock.txt  (RECORD these in results.md)')

## Phase 6 (early) — unit tests
These gate the model code and need **no data** — run them before downloading tens of GB.

In [ ]:
!python tests/test_numpy_ref.py    # mirror + PE math (torch-free)
!python tests/test_sas.py          # the 4 required SAS tests + numpy cross-checks
!python tests/test_tv_model.py     # RetinaNet + SAS wiring, weight sharing, train/infer

## One-time — reclaim Drive space from earlier runs

Do this **now**, before Phase 3 writes the dataset to Drive — prep needs ~350 MB of free space and
will fail if the quota is still full.

Earlier sessions pointed `--work-dir` at Drive, so each pruned checkpoint (~300 MB/epoch) was moved
to Drive's **Trash**, where it *still counts against the 15 GB quota* for 30 days. A 24-epoch run
strands ~7 GB that way. The cells below never do this again — weights now stay on ephemeral
`/content` — but the existing mess needs clearing once, **in this order**:

1. **Run the next cell.** It deletes any leftover `.pth` still sitting in `tb_runs` on Drive
   (deleting only moves them to Trash — that's step 2's job).
2. **Empty the Trash by hand** — this cannot be done from Colab, since the Drive mount has no
   empty-trash operation and `rm` on the mount is what *creates* trash:
   > **[drive.google.com/drive/trash](https://drive.google.com/drive/trash) → "Empty trash"**
3. **Check the quota:** [drive.google.com/settings/storage](https://drive.google.com/settings/storage).
   The 15 GB is shared across **Drive + Gmail + Photos** — if it still looks full, the culprit may
   not be Drive at all.

Do this once and it stays done.

In [ ]:
# One-time: delete leftover checkpoints that earlier runs wrote onto Drive.
# Run this BEFORE Phase 3 prep — prep writes ~350MB to Drive and will fail if the quota is full.
# Logs are KEPT — they're the actual results and cost nothing.
import glob, os

stale = glob.glob(f'{RUNS}/**/*.pth', recursive=True)
if not stale:
    print(f'No stale .pth on Drive under {RUNS} — nothing to clean.')
else:
    total = sum(os.path.getsize(p) for p in stale)
    print(f'Found {len(stale)} leftover checkpoint(s) on Drive, {total/1e9:.2f} GB:')
    for p in stale:
        print(f'   {os.path.getsize(p)/1e6:7.0f} MB  {p}')
    for p in stale:
        os.remove(p)          # -> Drive Trash; the manual Empty Trash step actually frees the quota
    print(f'\nDeleted. They are now in Drive Trash and STILL count against your quota.')
    print('   -> Empty it at https://drive.google.com/drive/trash to reclaim the space.')

kept = glob.glob(f'{RUNS}/**/*.jsonl', recursive=True)
print(f'\nKept {len(kept)} log file(s) (the results):')
for p in kept:
    print(f'   {os.path.getsize(p)/1e3:7.1f} KB  {p}')

## Phase 2 — download the raw dataset
Goes to **ephemeral `/content`**, never Drive. Skipped automatically if the compact set already
exists on Drive from a previous session.

If `gdown` fails (Google Drive quota on large public files is common), download the archive
manually from https://github.com/yun-liu/Tuberculosis and upload/extract it to `/content`.

In [ ]:
import os, glob, shutil

# Official TBX11K Google Drive file id (from github.com/yun-liu/Tuberculosis).
# Pass the BARE ID positionally — every gdown version accepts `url_or_id`, whereas `--fuzzy`
# (needed only to parse a full /view URL) does not exist in older gdown builds.
TBX11K_GDRIVE_ID = '1r-oNYTPiPCOUzSjChjCIYTdkjBTugqxR'
ZIP = '/content/tbx11k.zip'

def compact_ready(p):
    """True only if the compact dataset REALLY exists — never trust a bare directory.
    A failed earlier run can leave empty dirs behind that otherwise look 'done'."""
    return (os.path.isfile(f'{p}/annotations/tb_train_agnostic.json')
            and os.path.isfile(f'{p}/annotations/tb_val_agnostic.json')
            and os.path.isdir(f'{p}/images/train')
            and len(os.listdir(f'{p}/images/train')) > 0)

# --- clean up stale leftovers from any previous failed run ---
if os.path.isdir(DATA_512) and not compact_ready(DATA_512):
    print('removing stale/incomplete compact dir:', DATA_512)
    shutil.rmtree(DATA_512, ignore_errors=True)
if os.path.isdir(DATA_RAW) and not os.listdir(DATA_RAW):
    print('removing empty leftover raw dir:', DATA_RAW)
    os.rmdir(DATA_RAW)

if compact_ready(DATA_512):
    print('compact dataset already on Drive ->', DATA_512)
    print('Nothing to download. Skip to Phase 4.')
else:
    have_raw = os.path.isdir(DATA_RAW) and os.listdir(DATA_RAW)
    if not have_raw:
        print('Downloading TBX11K (tens of GB) to ephemeral /content ...')
        !pip install -q -U gdown
        !gdown {TBX11K_GDRIVE_ID} -O {ZIP}
        if os.path.isfile(ZIP) and os.path.getsize(ZIP) > 10_000_000:
            print(f'\ngot archive: {os.path.getsize(ZIP)/1e9:.1f} GB — unzipping ...')
            !unzip -q {ZIP} -d /content && rm -f {ZIP}
        else:
            if os.path.isfile(ZIP):
                # a tiny "archive" is almost always Drive's HTML quota/scan page
                print('\n!! downloaded file is too small to be the dataset — it is probably '
                      'Google Drive\'s quota / virus-scan HTML page:')
                !head -c 400 {ZIP}
                os.remove(ZIP)
            print('\n!! No usable archive. Google Drive throttles large public files.'
                  '\n   Options:'
                  '\n     1. Open the link in a browser, copy it to YOUR Drive, then use'
                  '\n        gdown with your own copy\'s id (personal copies are not throttled):'
                  '\n        https://drive.google.com/file/d/' + TBX11K_GDRIVE_ID + '/view'
                  '\n     2. Or download locally and upload/extract under /content.'
                  '\n   Then re-run this cell.')
    print('\nTop-level dirs under /content (find the extracted dataset root):')
    for d in sorted(glob.glob('/content/*')):
        if os.path.isdir(d) and os.path.basename(d) not in ('drive', 'sample_data', 'FOP'):
            try:
                n = len(os.listdir(d))
            except OSError:
                n = -1
            print(f'    {d}   ({n} entries)')
!df -h /content | tail -1

## Phase 2 GATE — inspect the actual layout
The official README documents the download links but **not** the folder structure, so we discover
it rather than assume it. Check the report: XML count, image count, and that every class name maps
to active/latent. Expect ~1,200 TB images (train ~600 / val ~200 per paper Table 2).

In [ ]:
# If the listing above shows a different folder name, point DATA_RAW at it here.
DATA_RAW = '/content/TBX11K'
!python tools/prepare_tbx11k.py --inspect --src {DATA_RAW}

## Phase 3 — build the compact TB-only 512² COCO set → Drive
Layout is auto-discovered. Override with `--xml-dir` / `--train-list` / `--val-list` if the report
above shows something unexpected. This is the only thing that gets persisted.

In [ ]:
!python tools/prepare_tbx11k.py --selftest    # verify the prep logic first (synthetic)
!python tools/prepare_tbx11k.py --src {DATA_RAW} --dst {DATA_512} --size 512 --write-agnostic
!du -sh {DATA_512}   # expect a few hundred MB

In [ ]:
# GATE: sanity-check a few overlays before training on this data.
import json, matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, matplotlib.patches as patches
from PIL import Image
d = json.load(open(f'{DATA_512}/annotations/tb_train.json'))
by_img = {}
for a in d['annotations']:
    by_img.setdefault(a['image_id'], []).append(a)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, im in zip(axes, d['images'][:4]):
    ax.imshow(Image.open(f"{DATA_512}/images/train/{im['file_name']}"), cmap='gray')
    for a in by_img.get(im['id'], []):
        x, y, w, h = a['bbox']
        ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, edgecolor='lime', lw=2))
    ax.set_title(f"{im['file_name'][:18]}  ({len(by_img.get(im['id'], []))} box)"); ax.axis('off')
plt.tight_layout(); plt.savefig('/content/overlay_check.png', dpi=70)
from IPython.display import Image as IPImage, display; display(IPImage('/content/overlay_check.png'))
print(f"train images={len(d['images'])} boxes={len(d['annotations'])}")

## Phase 4 — smoke test (a few batches, 1 epoch)

In [ ]:
!python tools/tv_train.py --work-dir {WORK}/smoke --data-root {DATA_512}/ --no-sas \
    --epochs 1 --batch-size 2 --limit-batches 5 --drive-sync {RUNS}/smoke
!python tools/tv_eval.py --ckpt {WORK}/smoke/epoch_1.pth --data-root {DATA_512}/ \
    --drive-sync {RUNS}/smoke
# GATE: runs end-to-end and prints a finite AP (~0 is expected and fine).

## Phase 5 — RetinaNet baseline (re-run after any time-out; it auto-resumes)

In [ ]:
# Weights -> /content/work (ephemeral). Only the tiny logs go to Drive.
!python tools/tv_train.py --work-dir {WORK}/baseline --data-root {DATA_512}/ --no-sas \
    --epochs 24 --batch-size 8 --lr 0.005 --seed 0 --drive-sync {RUNS}/baseline

In [ ]:
!python tools/tv_eval.py --ckpt {WORK}/baseline/epoch_24.pth --data-root {DATA_512}/ \
    --drive-sync {RUNS}/baseline

## Phase 7 — SymFormer w/ RetinaNet (the primary comparison)

In [ ]:
# --sync-weights keeps ONE model (best.pth, ~300MB) on Drive — this is the headline run, so it's
# the one worth having for a figure/demo later. Drop the flag if Drive is tight; the AP numbers in
# the logs are what the claim rests on, not the weights.
!python tools/tv_train.py --work-dir {WORK}/symformer --data-root {DATA_512}/ \
    --attention symattention --pe spe --stn --direction r2l \
    --epochs 24 --batch-size 8 --lr 0.005 --seed 0 \
    --drive-sync {RUNS}/symformer --sync-weights

In [ ]:
!python tools/tv_eval.py --ckpt {WORK}/symformer/epoch_24.pth --data-root {DATA_512}/ \
    --drive-sync {RUNS}/symformer
# GATE (primary claim): SymFormer AP50 > baseline AP50. Record both in results.md.

## Phase 8 — Table 8 ablation (long; each cell auto-resumes within a session)
Weights go to ephemeral `/content/work` and are deleted after each cell is evaluated, so the VM
disk stays bounded. Only each cell's `eval_log.jsonl` (a few KB) reaches Drive — that's the table.

In [ ]:
import subprocess, glob, os
DELETE_CKPTS_AFTER_EVAL = True   # frees /content (~300MB/cell). Nothing here ever touches Drive.
EPOCHS = 24

# (name, extra flags) — most-informative cells first, so a partial run is still meaningful.
CELLS = [
    ('none_none',            ['--no-sas']),                                                  # baseline
    ('vanilla_ape',          ['--attention','vanilla','--pe','ape']),
    ('symattention_ape',     ['--attention','symattention','--pe','ape']),
    ('symattention_spe_stn_r2l',   ['--attention','symattention','--pe','spe','--stn','--direction','r2l']),
    ('symattention_spe_nostn_r2l', ['--attention','symattention','--pe','spe','--no-stn','--direction','r2l']),
    ('symattention_spe_stn_l2r',   ['--attention','symattention','--pe','spe','--stn','--direction','l2r']),
    ('vanilla_rpe',          ['--attention','vanilla','--pe','rpe']),
    ('symattention_rpe',     ['--attention','symattention','--pe','rpe']),
    ('vanilla_spe_nostn_l2r',['--attention','vanilla','--pe','spe','--no-stn','--direction','l2r']),
    ('vanilla_spe_nostn_r2l',['--attention','vanilla','--pe','spe','--no-stn','--direction','r2l']),
    ('vanilla_spe_stn_l2r',  ['--attention','vanilla','--pe','spe','--stn','--direction','l2r']),
    ('vanilla_spe_stn_r2l',  ['--attention','vanilla','--pe','spe','--stn','--direction','r2l']),
    ('symattention_spe_nostn_l2r', ['--attention','symattention','--pe','spe','--no-stn','--direction','l2r']),
]

for name, flags in CELLS:
    wd  = f'{WORK}/abl/{name}'        # weights: ephemeral
    log = f'{RUNS}/abl/{name}'        # logs: Drive (KB)
    print('='*70); print('TRAIN', name); print('='*70)
    subprocess.run(['python','tools/tv_train.py','--work-dir',wd,'--data-root',f'{DATA_512}/',
                    '--epochs',str(EPOCHS),'--batch-size','8','--lr','0.005','--seed','0',
                    '--drive-sync',log] + flags)
    ck = f'{wd}/epoch_{EPOCHS}.pth'
    if os.path.isfile(ck):
        print('EVAL', name)
        subprocess.run(['python','tools/tv_eval.py','--ckpt',ck,'--data-root',f'{DATA_512}/',
                        '--tag',name,'--drive-sync',log])
        if DELETE_CKPTS_AFTER_EVAL:
            for p in glob.glob(f'{wd}/*.pth'):
                os.remove(p)
            print('removed checkpoints for', name)
    else:
        print('!! no checkpoint for', name, '— training did not finish; re-run this cell to resume')

print('\nlogs on Drive (expect KB):')
!du -sh {RUNS}
print('weights on the VM (ephemeral):')
!du -sh {WORK}

In [ ]:
# Collect every eval_log.jsonl into one table for results.md
import glob, json
rows = []
for f in sorted(glob.glob(f'{RUNS}/**/eval_log.jsonl', recursive=True)):
    for line in open(f):
        rows.append(json.loads(line))
print(f"{'run':32s} {'AP50':>6s} {'AP':>6s}")
for r in rows:
    print(f"{r['tag']:32s} {r['AP50']:6.1f} {r['AP']:6.1f}")

## Phase 10 — write-up
Fill `results.md` (baseline vs SymFormer + the ablation table), update `CLAUDE.md` §0/§6/§8 with
the actual numbers, and note deviations (torchvision instead of mmdet, batch size, versions) and
caveats (val-not-test; reduced data → trend not absolute numbers). Then commit and push.